# NimbusPay Ticket Triage — LoRA fine-tune of Qwen2.5-1.5B-Instruct

**Base model:** `Qwen/Qwen2.5-1.5B-Instruct` (1.54B params, under the 4B cap)
**Method:** LoRA (16-bit) and QLoRA (4-bit NF4) via PEFT
**Hardware target:** free Colab T4 (16 GB, Turing / compute 7.5 — **fp16 only, no bf16**)

This notebook runs top to bottom and reproduces the final adapter and `predictions_test.jsonl`.

| Part | What it does |
|---|---|
| A | Audit `train.jsonl`, emit `cleaning_log.csv` + `train_clean.jsonl` |
| B | Two baselines on dev: fixed prompt, and rules-in-prompt (reference only) |
| C | Fine-tuning pipeline: chat template, assistant-only loss, evidence-based seq length |
| D | Four controlled experiments, one variable at a time |
| E | Error analysis on dev with the best model |
| — | Test predictions |

**Why this stack.** `transformers.Trainer` + `peft` + `bitsandbytes`, with the tokenization and the
loss mask written out by hand.

I started with TRL's `SFTTrainer`, which is the obvious choice, and moved off it deliberately. TRL
masks the prompt by string-matching a `response_template` against the tokenized sequence; it works,
but when it silently fails to match you get a model trained to reproduce support tickets, a loss
curve that still looks fine, and no error. Since "loss on the right tokens" is one of the things
this task is explicitly judged on, I wanted it to be a visible assertion rather than a library
default I was trusting. Part C below builds the mask directly and asserts that the unmasked span
decodes back to exactly the label.

`peft` + `bitsandbytes` lets me flip between 16-bit LoRA and 4-bit QLoRA with one argument, which is
what experiment 2 needs, and `save_pretrained` emits a standard PEFT adapter directory with no
custom save path. I did not use Unsloth: it is faster on a T4, but it patches the model classes, and
I would rather have the chat template and the mask plainly inspectable than hidden behind a wrapper.

## 0. Environment

`EXPERIMENT MODE` note: the four controlled experiments in Part D each train a model.
Running all of them plus the final model exceeds the 3-hour budget for a single fresh runtime,
which the assignment explicitly allows ("other experiments may be shown from saved logs, but
include their code"). So:

* `RUN_EXPERIMENTS = False` (default) — runs baselines, the **final** model, eval and test
  predictions live, and loads the Part D table from `experiments_results.json`.
* `RUN_EXPERIMENTS = True` — reruns all four experiments from scratch and regenerates that JSON.

Data files are uploaded directly into `/content` and all artifacts are written to `/content/out`,
which the last cell zips for download. Nothing touches Google Drive.

In [1]:
import subprocess, sys
# Colab ships Python 3.13 with a CUDA-enabled torch already installed. Pinning torch (e.g. 2.4.1)
# fails outright here: there is no cp313 wheel for it, and the whole install aborts. So keep
# Colab's torch and install only the training stack on top. The exact resolved versions are
# printed in the next cell, which is what actually makes the run reproducible.
PKGS = ["transformers>=4.46,<5", "peft>=0.13,<0.18", "trl>=0.12,<0.24",
        "datasets>=3.0", "accelerate>=1.1", "bitsandbytes>=0.45"]
subprocess.run([sys.executable, "-m", "pip", "install", "-q"] + PKGS, check=True)
print("installed")

installed


In [2]:
import os, json, re, csv, math, time, random, gc, datetime, shutil
from collections import Counter, defaultdict, OrderedDict
import numpy as np, torch, transformers, peft, trl

SEED = 3407
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)

print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| peft", peft.__version__, "| trl", trl.__version__)
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > T4 GPU."
CAP = torch.cuda.get_device_capability()
print("GPU:", torch.cuda.get_device_name(0), "| capability", CAP,
      "| VRAM %.1f GB" % (torch.cuda.get_device_properties(0).total_memory / 1e9))
# Turing (7.5) has no bf16. Everything below uses fp16; asserting it here stops a silent 10x slowdown.
SUPPORTS_BF16 = CAP[0] >= 8
print("bf16 supported:", SUPPORTS_BF16, "-> using fp16" if not SUPPORTS_BF16 else "")

RUN_EXPERIMENTS = False   # see note above
BASE_MODEL = "Qwen/Qwen2.5-1.5B-Instruct"

torch 2.14.1+cu130 | transformers 4.57.6 | peft 0.17.1 | trl 0.23.1
GPU: Tesla T4 | capability (7, 5) | VRAM 15.6 GB
bf16 supported: False -> using fp16


In [3]:
# --- locate the data. Files are uploaded straight into /content via the Colab Files panel,
# --- so there is no Drive mount here: drive.mount() opens a modal auth popup, and this notebook
# --- is driven through the UI.
import glob
SEARCH = ["/content", "/content/data", "data", "."]
DATA_DIR = None
for c in SEARCH:
    if os.path.isfile(os.path.join(c, "train.jsonl")):
        DATA_DIR = c; break
if DATA_DIR is None:
    hits = glob.glob("/content/**/train.jsonl", recursive=True)
    if hits: DATA_DIR = os.path.dirname(hits[0])
if DATA_DIR is None:
    from google.colab import files
    print("Upload train.jsonl, dev.jsonl and test_inputs.jsonl")
    up = files.upload()
    for name in up: print("  received", name)
    DATA_DIR = "/content"
print("DATA_DIR =", DATA_DIR)
for f in ("train.jsonl", "dev.jsonl", "test_inputs.jsonl"):
    pth = os.path.join(DATA_DIR, f)
    assert os.path.isfile(pth), f"missing {pth} - upload it into /content and re-run this cell"
    print(f"  {f:18} {os.path.getsize(pth)/1024:8.1f} KB")

# Artifacts live on the Colab VM. Nothing is written to Drive.
OUT_DIR = "/content/out"
os.makedirs(OUT_DIR, exist_ok=True)
print("OUT_DIR  =", OUT_DIR)
print("NOTE: this is VM-local. The final cell zips it for download, so grab the zip before the")
print("      runtime disconnects or the artifacts are lost.")

def load_jsonl(p):
    with open(p, encoding="utf-8") as f:
        return [json.loads(l) for l in f if l.strip()]

train_raw = load_jsonl(os.path.join(DATA_DIR, "train.jsonl"))
dev       = load_jsonl(os.path.join(DATA_DIR, "dev.jsonl"))
test      = load_jsonl(os.path.join(DATA_DIR, "test_inputs.jsonl"))
print(f"train={len(train_raw)} dev={len(dev)} test={len(test)}")

DATA_DIR = /content
  train.jsonl          1135.9 KB
  dev.jsonl             143.6 KB
  test_inputs.jsonl     231.0 KB
OUT_DIR  = /content/out
NOTE: this is VM-local. The final cell zips it for download, so grab the zip before the
      runtime disconnects or the artifacts are lost.
train=1445 dev=200 test=400


## Part A — Audit the training data

The method here matters more than the list of defects. I encoded `SCHEMA.md` as an **executable
validator**, then ran it against `dev.jsonl` first. Dev is stated to be clean and follows the spec
exactly, so dev is a **control**: if my validator flags a clean row, my validator is wrong, not the
data. I iterated on the validator until it emitted **zero flags across all 200 dev rows**.

Only then did I point it at `train.jsonl`. Every flag it raises there is therefore a real defect
rather than a misreading of the spec. Two of my initial hypotheses were wrong and both are reported
below as rejected, because an audit that only reports confirmations is not an audit.

In [4]:
KEYS   = ["category","priority","amount_paise","txn_date","txn_id","channel","language","needs_human"]
CATS   = {"payment_failed","refund","fraud","account_access","kyc","offers","other"}
CHANS  = {"upi","card","netbanking","wallet"}
NO_TXN = {"account_access","kyc","other"}   # spec: txn fields always null for these

# Legal-threat trigger for P1 (spec: RBI, ombudsman, consumer court, lawyer, legal notice/action).
LEGAL = re.compile(r"\b(rbi|ombudsman|consumer court|consumer forum|lawyer|advocate|"
                   r"legal notice|legal action|legal|court)\b", re.I)
# "already contacted support before" trigger for needs_human. These are the exact phrasings the
# generator uses; I derived the list from the dev rows where needs_human=True and priority!=P1.
REPEAT = re.compile(r"(again and again|baar[\s-]?baar|already written \d+ times|"
                    r"(second|third|fourth|2nd|3rd|4th)\s+complaint|"
                    r"(second|third|fourth|2nd|3rd|4th)\s+time\s+(i\s+am\s+)?contact|"
                    r"pehle bhi complaint|keep raising|dusri baar|teesri baar)", re.I)

def body_of(user_msg):
    """Ticket text without the Received/Source header AND without quoted-email lines.
    The spec says dates inside '>' lines are not the transaction date, so quoted blocks are
    distractors. They also contain 'already received a resolution', which false-fires REPEAT --
    that was the bug that made my first validator flag 16 clean dev rows."""
    b = user_msg.split("\n\n", 1)[1] if "\n\n" in user_msg else user_msg
    return "\n".join(l for l in b.split("\n") if not l.lstrip().startswith(">"))

def recv_of(user_msg):
    m = re.search(r"Received:\s*(\d{4}-\d{2}-\d{2})", user_msg)
    return datetime.date.fromisoformat(m.group(1)) if m else None

def priority_of(cat, amt, body):
    """SCHEMA.md priority ladder, applied in the specified order."""
    if cat == "fraud": return "P1"
    if amt is not None and amt >= 50_000 * 100: return "P1"
    if LEGAL.search(body): return "P1"
    if cat in ("payment_failed", "refund") and amt is not None and amt >= 5_000 * 100: return "P2"
    if cat == "account_access": return "P2"
    return "P3"

def needs_human_of(prio, body):
    return True if prio == "P1" else bool(REPEAT.search(body))

In [5]:
def validate(rows):
    """Return {defect_class: [evidence,...]} for one split."""
    F = defaultdict(list)
    for r in rows:
        rid, user, raw = r["id"], r["messages"][1]["content"], r["messages"][-1]["content"]
        body, recv = body_of(user), recv_of(user)
        try:
            o = json.loads(raw)
        except Exception as e:
            F["D1_broken_json"].append((rid, str(e)[:40])); continue
        if not isinstance(o, dict):
            F["D1_broken_json"].append((rid, "not a JSON object")); continue
        # --- structural: each checked independently so one defect cannot mask another ---
        if set(o) != set(KEYS):
            F["D3_wrong_keys"].append((rid, sorted(set(o) ^ set(KEYS))))
        td, ti, amt = o.get("txn_date"), o.get("txn_id"), o.get("amount_paise")
        if td is not None and not (isinstance(td, str) and re.fullmatch(r"\d{4}-\d{2}-\d{2}", td)):
            F["D4_date_format"].append((rid, td))
        if ti is not None and not (isinstance(ti, str) and re.fullmatch(r"NP\d{10}", ti)):
            F["D5_txnid_format"].append((rid, ti))
        if amt is not None and (isinstance(amt, bool) or not isinstance(amt, int)):
            F["Dx_amount_type"].append((rid, repr(amt)))
        if not isinstance(o.get("needs_human"), bool):
            F["Dx_needs_human_type"].append((rid, repr(o.get("needs_human"))))
        if o.get("category") not in CATS: F["Dx_bad_category"].append((rid, o.get("category")))
        if o.get("language") not in {"en","hi-en"}: F["Dx_bad_language"].append((rid, o.get("language")))
        if o.get("channel") is not None and o.get("channel") not in CHANS:
            F["Dx_bad_channel"].append((rid, o.get("channel")))
        # --- semantic rules, only where the fields are well-formed enough to judge ---
        cat, prio = o.get("category"), o.get("priority")
        if cat in NO_TXN and any(o.get(k) is not None for k in ("amount_paise","txn_date","txn_id","channel")):
            F["D9_txnfields_on_nontxn"].append((rid, cat))
        if cat in CATS and (amt is None or (isinstance(amt, int) and not isinstance(amt, bool))):
            w = priority_of(cat, amt, body)
            if prio != w: F["D7_priority_rule"].append((rid, cat, amt, prio, w))
            if isinstance(o.get("needs_human"), bool) and prio in {"P1","P2","P3"}:
                wn = needs_human_of(prio, body)
                if o["needs_human"] != wn: F["D8_needs_human_rule"].append((rid, o["needs_human"], wn, prio))
        if isinstance(td, str) and re.fullmatch(r"\d{4}-\d{2}-\d{2}", td) and recv:
            if datetime.date.fromisoformat(td) > recv:
                F["Dx_future_date"].append((rid, td, str(recv)))
        if isinstance(ti, str):
            digits = re.sub(r"\D", "", ti)
            if digits and digits not in re.sub(r"[^A-Z0-9]", "", user.upper()):
                F["Dx_txnid_absent_from_text"].append((rid, ti))
    return F

# --- THE CONTROL. This must print zero. ---
dev_flags = validate(dev)
print("dev.jsonl (known clean) flags:", {k: len(v) for k, v in dev_flags.items()} or "NONE")
assert not dev_flags, "Validator disagrees with a clean row -> fix the validator, not the data."
print("\nValidator reproduces SCHEMA.md exactly on 200/200 clean dev rows.")
print("Every train flag below is therefore a real defect.\n")

train_flags = validate(train_raw)
print(f"{'defect':30} {'count':>6}  {'% of 1445':>9}")
for k in sorted(train_flags, key=lambda k: -len(train_flags[k])):
    n = len(train_flags[k]); print(f"{k:30} {n:6d}  {n/len(train_raw):9.1%}")

dev.jsonl (known clean) flags: NONE

Validator reproduces SCHEMA.md exactly on 200/200 clean dev rows.
Every train flag below is therefore a real defect.

defect                          count  % of 1445
D7_priority_rule                  135       9.3%
D3_wrong_keys                      70       4.8%
D8_needs_human_rule                65       4.5%
D4_date_format                     30       2.1%
D1_broken_json                     25       1.7%
D5_txnid_format                    20       1.4%
Dx_txnid_absent_from_text           4       0.3%


### A.1 The legacy batch is a different spec version, not noise

70 rows carry the key `prio` instead of `priority`. The obvious move is to recompute their priority
from the rule. That would be wrong and would throw away 70 correct labels.

Their `prio` values are not `P1/P2/P3` at all — they are `low/medium/high`. Mapped against what the
v2 ladder derives for the same row, the correspondence is perfect with **zero exceptions**:

| legacy | n | v2 |
|---|---|---|
| `low` | 44 | `P3` |
| `medium` | 15 | `P2` |
| `high` | 11 | `P1` |

`SCHEMA.md` calls itself "v2, current", so this is a batch exported under v1 with the old enum
vocabulary. The right fix is a **value remap plus a key rename**, which preserves all 70 labels.
The same 70 rows also carry the `DD/MM/YYYY` dates (all 30 of them fall inside this batch), which
corroborates a single legacy exporter rather than three unrelated faults.

In [6]:
legacy, modern = [], []
for r in train_raw:
    try: o = json.loads(r["messages"][-1]["content"])
    except Exception: continue
    if "prio" in o and "priority" not in o:
        o = dict(o); o["priority"] = o.pop("prio"); legacy.append((r, o))
    elif set(o) == set(KEYS):
        modern.append((r, o))

pairs = Counter()
for r, o in legacy:
    body = body_of(r["messages"][1]["content"])
    amt = o.get("amount_paise")
    if o.get("category") in CATS and (amt is None or isinstance(amt, int)):
        pairs[(o["priority"], priority_of(o["category"], amt, body))] += 1
print("legacy batch n =", len(legacy))
print("(legacy value -> v2 value derived from the rule):")
for (got, want), n in pairs.most_common(): print(f"   {got:7} -> {want}   {n:3d}")
unexplained = sum(n for (g, w), n in pairs.items() if g not in ("low","medium","high"))
print("\nlegacy rows NOT explained by a pure enum remap:", unexplained)

dd = {x[0] for x in train_flags.get("D4_date_format", [])}
lg = {r["id"] for r, _ in legacy}
print(f"DD/MM/YYYY dates: {len(dd)}, of which inside the legacy batch: {len(dd & lg)}")

legacy batch n = 70
(legacy value -> v2 value derived from the rule):
   low     -> P3    44
   medium  -> P2    15
   high    -> P1    11

legacy rows NOT explained by a pure enum remap: 0
DD/MM/YYYY dates: 30, of which inside the legacy batch: 30


### A.2 Two hypotheses I tested and rejected

**Rejected: the annotator conflated customer urgency with priority.** 33 rows are labelled `P1`
where the ladder gives `P3`, and several contain "URGENT!!". If urgency language drove the error,
those rows should be enriched for it. They are not: 24% contain urgency markers versus 21% of all
other rows. No signal. The over-escalation is unexplained label noise, so I correct it by rule
rather than by a story about where it came from.

**Rejected: a bad annotator batch.** If defects came from one worker or one time window they would
cluster by id. Bucketed into ten ranges of ~145 ids, broken JSON, junk bodies, the legacy batch and
priority errors are all spread evenly (priority error rate 1.5%–7.3%, no trend). So there is no
contiguous block to excise; every fix has to be row-by-row.

**Also checked, nothing found:** amounts picked from a balance/limit/fee distractor instead of the
transaction (0 rows, in both dev and train); tickets with explicit fraud wording labelled as
something else (0 rows — `fraud` overrides every other category and that is respected throughout);
systematic `language` corruption (none — the two rows my detector flagged, "Registered email id
kaise change karun?", are genuinely Hinglish and correctly labelled, so those were my detector's
false positives). Duplicate ticket texts exist (60 groups, 120 rows) but **their labels always
agree**, so they are a reweighting problem, not a correctness one.

In [7]:
URG = re.compile(r"(URGENT|ASAP|immediately|jaldi|turant|emergency|!!)", re.I)
over, rest = [], []
for r, o in modern:
    body = body_of(r["messages"][1]["content"]); amt = o["amount_paise"]
    if o["category"] not in CATS or (amt is not None and not isinstance(amt, int)): continue
    w = priority_of(o["category"], amt, body)
    (over if (o["priority"] == "P1" and w == "P3") else rest).append(bool(URG.search(body)))
print("HYPOTHESIS: urgency language drives P1 over-escalation")
print(f"  over-escalated P1->P3 rows with urgency wording: {sum(over)}/{len(over)} = {sum(over)/len(over):.0%}")
print(f"  all other rows with urgency wording            : {sum(rest)}/{len(rest)} = {sum(rest)/len(rest):.0%}")
print("  VERDICT: rejected, no enrichment.\n")

def bucket(rid): return (int(rid.split("-")[1]) - 1) // 145
tot, err = Counter(), Counter()
for r, o in modern:
    b = bucket(r["id"]); tot[b] += 1
    body = body_of(r["messages"][1]["content"]); amt = o["amount_paise"]
    if o["category"] not in CATS or (amt is not None and not isinstance(amt, int)): continue
    if o["priority"] != priority_of(o["category"], amt, body): err[b] += 1
print("HYPOTHESIS: defects cluster in one id range (bad annotator batch)")
print("  bucket  n   prio-errors  rate")
for b in sorted(tot): print(f"  {b:5d} {tot[b]:4d} {err[b]:9d}   {err[b]/max(1,tot[b]):6.1%}")
print("  VERDICT: rejected, defects are uniform.")

HYPOTHESIS: urgency language drives P1 over-escalation
  over-escalated P1->P3 rows with urgency wording: 8/33 = 24%
  all other rows with urgency wording            : 275/1317 = 21%
  VERDICT: rejected, no enrichment.

HYPOTHESIS: defects cluster in one id range (bad annotator batch)
  bucket  n   prio-errors  rate
      0  134         7     5.2%
      1  134         7     5.2%
      2  138        10     7.2%
      3  136         8     5.9%
      4  133         6     4.5%
      5  135         7     5.2%
      6  135         5     3.7%
      7  134         8     6.0%
      8  139         2     1.4%
      9  132         5     3.8%
  VERDICT: rejected, defects are uniform.


### A.3 The cleaner

Staged, and the order is load-bearing:

1. **Drop** unparseable labels (nothing to recover) and placeholder bodies (`test`, `asdf`, one
   empty) whose labels are fabricated — all four rows whose `txn_id` does not appear anywhere in
   the ticket turn out to be these same junk rows.
2. **Fix** the legacy key and enum (A.1), then date and `txn_id` formats.
3. **Fix** the rupee/paise unit error **before** touching priority. This ordering is not cosmetic:
   `tr-00382` stores ₹46,094 as `46094` paise, which makes a *correct* `P2` look like a `P3`
   violation. Recomputing priority first would have "fixed" three correct labels into wrong ones.
4. **Fix** priority, then `needs_human` (which depends on the corrected priority).
5. **Drop** exact duplicate ticket texts, keeping the first.

Then an independent re-validation of the output, which must come back clean.

In [8]:
JUNK = re.compile(r"^(test|asdf|qwerty|lorem|foo|bar|aaa+|xxx+|\.|\s|\d)+$", re.I)
LEGACY_PRIO = {"low": "P3", "medium": "P2", "high": "P1"}
NUM = r"(\d[\d,]*(?:\.\d+)?)"

def amount_candidates(body):
    """Every money figure in the text, in paise. 'k' = thousand, 'lakh' = 100000."""
    out = set()
    for m in re.finditer(NUM + r"\s*(k|lakh|lakhs)\b", body, re.I):
        mult = 1000 if m.group(2).lower() == "k" else 100000
        out.add(round(float(m.group(1).replace(",", "")) * mult * 100))
    for m in re.finditer(r"(?:₹|Rs\.?|INR|rupees?|rupaye)\s*" + NUM, body, re.I):
        out.add(round(float(m.group(1).replace(",", "")) * 100))
    for m in re.finditer(NUM + r"\s*(?:rupees?|rupaye|/-)", body, re.I):
        out.add(round(float(m.group(1).replace(",", "")) * 100))
    return out

def clean_train(rows):
    log, kept, S = [], [], Counter()
    for r in rows:
        rid, user, raw = r["id"], r["messages"][1]["content"], r["messages"][-1]["content"]
        body = body_of(user)
        try:
            o = json.loads(raw)
        except Exception:
            log.append((rid, "drop", "D1 label JSON truncated/unparseable - label unrecoverable")); S["D1_broken_json"] += 1; continue
        if not isinstance(o, dict):
            log.append((rid, "drop", "D1 label is not a JSON object")); S["D1_broken_json"] += 1; continue
        if not body.strip() or JUNK.fullmatch(body.strip()):
            log.append((rid, "drop", f"D2 placeholder/empty ticket body ({body.strip()[:16]!r}) with fabricated label")); S["D2_junk_body"] += 1; continue
        o = dict(o)
        # 2. legacy schema
        if "prio" in o and "priority" not in o:
            v = o.pop("prio"); S["D3_prio_key"] += 1
            log.append((rid, "fix", "D3 legacy key 'prio' -> 'priority'"))
            if isinstance(v, str) and v.lower() in LEGACY_PRIO:
                nv = LEGACY_PRIO[v.lower()]
                log.append((rid, "fix", f"D3b legacy priority enum remapped ({v} -> {nv})")); S["D3b_prio_enum"] += 1; v = nv
            o["priority"] = v
        td = o.get("txn_date")
        if isinstance(td, str) and not re.fullmatch(r"\d{4}-\d{2}-\d{2}", td):
            m = re.fullmatch(r"(\d{2})/(\d{2})/(\d{4})", td)
            if m:
                o["txn_date"] = f"{m.group(3)}-{m.group(2)}-{m.group(1)}"
                log.append((rid, "fix", f"D4 txn_date DD/MM/YYYY -> ISO ({td} -> {o['txn_date']})"))
            else:
                o["txn_date"] = None; log.append((rid, "fix", f"D4 txn_date unparseable ({td!r}) -> null"))
            S["D4_date_fmt"] += 1
        ti = o.get("txn_id")
        if isinstance(ti, str) and not re.fullmatch(r"NP\d{10}", ti):
            n = re.sub(r"[^A-Za-z0-9]", "", ti).upper()
            if re.fullmatch(r"NP\d{10}", n):
                o["txn_id"] = n; log.append((rid, "fix", f"D5 txn_id normalised ({ti!r} -> {n})"))
            else:
                o["txn_id"] = None; log.append((rid, "fix", f"D5 txn_id invalid ({ti!r}) -> null"))
            S["D5_txnid_fmt"] += 1
        # 3. unit error BEFORE priority
        amt = o.get("amount_paise")
        if isinstance(amt, int) and not isinstance(amt, bool):
            C = amount_candidates(body)
            if C and amt not in C and amt * 100 in C:
                o["amount_paise"] = amt * 100
                log.append((rid, "fix", f"D6 amount stored in rupees not paise ({amt} -> {amt*100})")); S["D6_amt_unit"] += 1
        # 4. derived fields
        cat, amt = o.get("category"), o.get("amount_paise")
        if cat in CATS and (amt is None or (isinstance(amt, int) and not isinstance(amt, bool))):
            w = priority_of(cat, amt, body)
            if o.get("priority") != w:
                log.append((rid, "fix", f"D7 priority violates spec ({o.get('priority')} -> {w}; cat={cat} amt={amt})"))
                o["priority"] = w; S["D7_priority"] += 1
            wn = needs_human_of(o["priority"], body)
            if o.get("needs_human") != wn:
                log.append((rid, "fix", f"D8 needs_human violates spec ({o.get('needs_human')} -> {wn}; prio={o['priority']})"))
                o["needs_human"] = wn; S["D8_needs_human"] += 1
        if cat in NO_TXN:
            for k in ("amount_paise","txn_date","txn_id","channel"):
                if o.get(k) is not None:
                    log.append((rid, "fix", f"D9 {k} must be null for category={cat} (was {o[k]!r})")); o[k] = None; S["D9_txnfields"] += 1
        r2 = json.loads(json.dumps(r))
        r2["messages"][-1]["content"] = json.dumps(OrderedDict((k, o[k]) for k in KEYS), ensure_ascii=False)
        kept.append(r2)
    # 5. dedupe
    seen, final = {}, []
    for r in kept:
        t = r["messages"][1]["content"]
        if t in seen:
            log.append((r["id"], "drop", f"D10 exact duplicate ticket text of {seen[t]} (labels agree)")); S["D10_duplicate"] += 1
        else:
            seen[t] = r["id"]; final.append(r)
    return final, log, S

train_clean, clean_log, stats = clean_train(train_raw)
print(f"raw {len(train_raw)} -> clean {len(train_clean)}")
print(f"dropped ids {len({l[0] for l in clean_log if l[1]=='drop'})} | fixed ids {len({l[0] for l in clean_log if l[1]=='fix'})} | log lines {len(clean_log)}")
for k, v in sorted(stats.items()): print(f"   {k:18} {v:5d}")

clean_log.sort(key=lambda x: (x[0], x[1], x[2]))
with open(os.path.join(OUT_DIR, "cleaning_log.csv"), "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f); w.writerow(["id","action","reason"]); w.writerows(clean_log)
with open(os.path.join(OUT_DIR, "train_clean.jsonl"), "w", encoding="utf-8") as f:
    for r in train_clean: f.write(json.dumps(r, ensure_ascii=False) + "\n")

# --- validation gate: the cleaned set must satisfy the same validator that dev satisfies ---
post = validate(train_clean)
print("\npost-clean validator flags:", {k: len(v) for k, v in post.items()} or "ZERO - cleaned set is fully spec-consistent")
assert not post, "cleaning left spec violations behind"

raw 1445 -> clean 1350
dropped ids 95 | fixed ids 191 | log lines 388
   D10_duplicate         60
   D1_broken_json        25
   D2_junk_body          10
   D3_prio_key           70
   D3b_prio_enum         70
   D4_date_fmt           30
   D5_txnid_fmt          20
   D6_amt_unit           20
   D7_priority           62
   D8_needs_human        21

post-clean validator flags: ZERO - cleaned set is fully spec-consistent


## Part B — Baselines on dev

Both baselines use the **exact inference contract** the graders will use: the `system` and `user`
messages verbatim, the model's own chat template, greedy decoding, `max_new_tokens=200`, and the
output taken as-is apart from stripping whitespace. The only difference between the two is the
system message.

Baseline 2 (rules in the prompt) is a reference point only — the assignment forbids it at test time.
It tells me how much of the gap is "the model cannot read the rules" versus "the model cannot do the
task at all", which is what decides whether fine-tuning is even the right lever.

In [9]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None: tok.pad_token = tok.eos_token
print("pad:", repr(tok.pad_token), "| eos:", repr(tok.eos_token))
print("\n--- chat template applied to a real row (add_generation_prompt=True) ---")
demo = tok.apply_chat_template(dev[0]["messages"][:2], tokenize=False, add_generation_prompt=True)
print(demo)
print("--- end ---")
# Qwen2.5 is ChatML and has a real system role, so the fixed system prompt is passed through
# untouched. No fallback needed.
assert "<|im_start|>system" in demo and demo.rstrip().endswith("<|im_start|>assistant")

/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

pad: '<|endoftext|>' | eos: '<|im_end|>'

--- chat template applied to a real row (add_generation_prompt=True) ---
<|im_start|>system
You are NimbusPay's ticket triage model. Read the ticket and return the triage JSON.<|im_end|>
<|im_start|>user
Received: 2026-08-28 (Fri)
Source: app-chat

Good morning, What needs to be done to activate and use a card? My earlier ticket number is #54047. Please look into it.<|im_end|>
<|im_start|>assistant

--- end ---


In [10]:
# --- sequence-length evidence ---
def full_len(msgs):
    return len(tok.apply_chat_template(msgs, tokenize=True, add_generation_prompt=False))
def prompt_len(msgs):
    return len(tok.apply_chat_template(msgs[:2], tokenize=True, add_generation_prompt=True))

tl = sorted(full_len(r["messages"]) for r in train_clean)
dl = sorted(prompt_len(r["messages"]) for r in dev)
sl = sorted(prompt_len(r["messages"]) for r in test)
def pct(a, p): return a[min(len(a) - 1, int(len(a) * p))]
print(f"train full (prompt+label) tokens: median={pct(tl,.5)} p90={pct(tl,.9)} p99={pct(tl,.99)} max={tl[-1]}")
print(f"dev   prompt tokens            : median={pct(dl,.5)} p90={pct(dl,.9)} p99={pct(dl,.99)} max={dl[-1]}")
print(f"test  prompt tokens            : median={pct(sl,.5)} p90={pct(sl,.9)} p99={pct(sl,.99)} max={sl[-1]}")
n_long = sum(1 for x in tl if x > 2 * pct(tl, .9))
print(f"\nThe distribution is bimodal: {n_long} train rows ({n_long/len(tl):.1%}) are far out in the tail.")
print("Those are the rows carrying quoted '>' email history (up to 17 repeated support replies).")

# Choose a length that cannot truncate a label. Right-truncation would amputate the assistant JSON,
# which sits at the END of the sequence -- it would silently corrupt the target, not just the input.
MAX_SEQ_LEN = int(min(2560, 64 * math.ceil(max(tl[-1], sl[-1] + 80) / 64)))
print(f"\nMAX_SEQ_LEN = {MAX_SEQ_LEN}  (covers the longest train sequence and the longest test prompt + 80 label tokens)")
print("Rejected alternative: a shorter length with the long rows dropped. The test set contains")
print("long quoted-email rows too, so the model has to have seen that shape in training.")
print("Cost is controlled with group_by_length=True instead, so the tail does not pad every batch.")

train full (prompt+label) tokens: median=168 p90=214 p99=1936 max=2215
dev   prompt tokens            : median=106 p90=136 p99=2106 max=2115
test  prompt tokens            : median=108 p90=143 p99=1849 max=2089

The distribution is bimodal: 38 train rows (2.8%) are far out in the tail.
Those are the rows carrying quoted '>' email history (up to 17 repeated support replies).

MAX_SEQ_LEN = 2240  (covers the longest train sequence and the longest test prompt + 80 label tokens)
Rejected alternative: a shorter length with the long rows dropped. The test set contains
long quoted-email rows too, so the model has to have seen that shape in training.
Cost is controlled with group_by_length=True instead, so the tail does not pad every batch.


In [11]:
FIXED_SYSTEM = dev[0]["messages"][0]["content"]
print("fixed system prompt:", repr(FIXED_SYSTEM))

RULES_SYSTEM = FIXED_SYSTEM + "\n\n" + """Return ONLY a JSON object with exactly these 8 keys: category, priority, amount_paise, txn_date, txn_id, channel, language, needs_human. Use null when a value is not present.
category: one of payment_failed, refund, fraud, account_access, kyc, offers, other. Use fraud when the customer did not make/approve/recognise the transaction (fraud wins over every other category). payment_failed when a payment they made failed, was declined or is stuck. refund when money is owed back (refund, cancellation, return, double charge). account_access for locked account, blocked PIN, forgotten passcode, OTP not received. kyc for identity verification, Aadhaar/PAN, documents. offers for cashback, promo codes, reward points, referral bonus. other for anything else.
priority, in this order: P1 if category is fraud, or the amount is 50000 rupees or more, or the customer makes a legal threat (RBI, ombudsman, consumer court, lawyer, legal notice/action). Else P2 if category is payment_failed or refund and the amount is 5000 rupees or more, or category is account_access. Else P3.
amount_paise: the amount of the transaction the ticket is about, in paise (1 rupee = 100 paise), as an integer. Ignore balances, limits and fees. k = thousand, lakh = 100000.
txn_date: YYYY-MM-DD. Resolve relative dates (yesterday, 3 days ago, kal, parso) against the Received date. If the year is missing use the most recent such date on or before Received. Dates inside quoted lines starting with > are NOT the transaction date.
txn_id: NP followed by 10 digits, uppercase, no spaces or symbols. Ticket, order and bank reference numbers are not transaction ids.
channel: upi, card, netbanking or wallet, only when explicitly stated.
language: hi-en if the message is Hinglish (Hindi in Latin script), else en.
needs_human: true if priority is P1, or the customer says they contacted support before. Else false.
For account_access, kyc and other, amount_paise, txn_date, txn_id and channel are ALWAYS null."""

@torch.no_grad()
def generate(model, rows, system=None, batch_size=8, max_new_tokens=200, desc=""):
    """The graders' inference contract: verbatim messages, model chat template, greedy, 200 tokens.
    Left padding so that batched generation is identical to one-at-a-time generation."""
    model.eval()
    old_side = tok.padding_side; tok.padding_side = "left"
    out = []
    t0 = time.time()
    for i in range(0, len(rows), batch_size):
        chunk = rows[i:i + batch_size]
        prompts = []
        for r in chunk:
            msgs = [dict(m) for m in r["messages"][:2]]
            if system is not None: msgs[0]["content"] = system
            prompts.append(tok.apply_chat_template(msgs, tokenize=False, add_generation_prompt=True))
        enc = tok(prompts, return_tensors="pt", padding=True, add_special_tokens=False).to(model.device)
        gen = model.generate(**enc, max_new_tokens=max_new_tokens, do_sample=False, num_beams=1,
                             temperature=None, top_p=None, top_k=None,
                             pad_token_id=tok.pad_token_id, eos_token_id=tok.eos_token_id)
        for r, g in zip(chunk, gen):
            text = tok.decode(g[enc["input_ids"].shape[1]:], skip_special_tokens=True)
            out.append({"id": r["id"], "output": text.strip()})   # strip() is the only post-processing
        if i == 0 or (i // batch_size) % 5 == 0:
            print(f"  {desc} {min(i+batch_size,len(rows))}/{len(rows)}  {time.time()-t0:.0f}s", flush=True)
    tok.padding_side = old_side
    return out

fixed system prompt: "You are NimbusPay's ticket triage model. Read the ticket and return the triage JSON."


In [12]:
# --- scorer: logic copied verbatim from the provided score.py so dev numbers are comparable ---
SCORE_KEYS = KEYS
def _same(a, b): return type(a) is type(b) and a == b
def _parse(text):
    try: obj = json.loads(text.strip())
    except (json.JSONDecodeError, AttributeError): return None
    return obj if isinstance(obj, dict) else None

def score(gold_rows, pred_rows):
    preds = {p["id"]: p.get("output") for p in pred_rows}
    groups = defaultdict(lambda: {"n":0,"valid":0,"exact":0,"field":defaultdict(int)})
    for g in gold_rows:
        gold = json.loads(g["messages"][-1]["content"])
        pred = _parse(preds.get(g["id"]))
        names = ["all"] + ([f"slice={g['slice']}"] if "slice" in g else []) + [f"category={gold['category']}"]
        hits = {k: pred is not None and k in pred and _same(pred[k], gold[k]) for k in SCORE_KEYS}
        exact = pred is not None and all(hits.values()) and set(pred) == set(SCORE_KEYS)
        for nm in names:
            s = groups[nm]; s["n"] += 1; s["valid"] += pred is not None; s["exact"] += exact
            for k in SCORE_KEYS: s["field"][k] += hits[k]
    out = {}
    for nm, s in groups.items():
        n = s["n"]; fields = {k: s["field"][k]/n for k in SCORE_KEYS}
        out[nm] = {"n":n, "json_valid":s["valid"]/n, "exact_match":s["exact"]/n,
                   "mean_field_acc": sum(fields.values())/len(SCORE_KEYS), "fields":fields}
    return out

def report(res, title):
    a = res["all"]
    print(f"\n=== {title} ===")
    print(f"  n={a['n']}  json_valid={a['json_valid']:.1%}  exact_match={a['exact_match']:.1%}  mean_field_acc={a['mean_field_acc']:.1%}")
    print("  per-field: " + "  ".join(f"{k}={v:.0%}" for k, v in a["fields"].items()))
    return a

In [13]:
def load_base(four_bit):
    """Load the base model. T4 is Turing: fp16 compute, never bf16.

    attn_implementation is load-bearing. With "eager", Qwen2's large activation outliers overflow
    fp16 inside the attention softmax and the forward pass returns NaN logits. argmax of NaN is
    index 0, which in Qwen's vocab is "!", so generation comes out as "!!!!!!!!..." and every
    baseline silently scores 0% json_valid. "sdpa" keeps the softmax numerically stable.
    The assertion below turns that failure from a plausible-looking 0% result into a loud error."""
    kw = dict(torch_dtype=torch.float16, device_map={"": 0}, attn_implementation="sdpa")
    if four_bit:
        kw["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type="nf4",
            bnb_4bit_compute_dtype=torch.float16, bnb_4bit_use_double_quant=True)
    m = AutoModelForCausalLM.from_pretrained(BASE_MODEL, **kw)
    m.config.use_cache = True
    with torch.no_grad():
        probe = tok("Received: 2026-01-01 Source: email My payment failed. Rs 1,200.",
                    return_tensors="pt").to(m.device)
        lg = m(**probe).logits
    assert torch.isfinite(lg).all(), (
        "non-finite logits from the base model - fp16 attention overflow. "
        "Generation would be all '!' (token 0) and every score would read 0%.")
    print("  base loaded, logits finite, %.2f GB" % (torch.cuda.memory_allocated() / 1e9))
    return m

def free(*objs):
    for o in objs:
        try: del o
        except Exception: pass
    gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()

free()
base = load_base(four_bit=False)

t0 = time.time()
b1 = generate(base, dev, system=FIXED_SYSTEM, desc="baseline-1")
r_b1 = report(score(dev, b1), "B1: base model, fixed prompt, no fine-tuning")
print("  minutes:", round((time.time()-t0)/60, 1))
print("\n  three raw outputs:")
for p in b1[:3]: print("   ", repr(p["output"][:200]))

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

  base loaded, logits finite, 3.10 GB
  baseline-1 8/200  13s
  baseline-1 48/200  46s
  baseline-1 88/200  86s
  baseline-1 128/200  128s
  baseline-1 168/200  172s

=== B1: base model, fixed prompt, no fine-tuning ===
  n=200  json_valid=0.0%  exact_match=0.0%  mean_field_acc=0.0%
  per-field: category=0%  priority=0%  amount_paise=0%  txn_date=0%  txn_id=0%  channel=0%  language=0%  needs_human=0%
  minutes: 3.4

  three raw outputs:
    '```json\n{\n  "status": "New",\n  "priority": "High",\n  "category": "Card Activation",\n  "description": "User received an activation request for their card but has not yet been able to use it. They need '
    '```json\n{\n  "priority": "normal",\n  "category": "customer_service",\n  "description": "Customer asks about accepted IDs for their device.",\n  "resolution_required": true,\n  "expected_resolution_time": '
    '```json\n{\n  "status": "urgent",\n  "priority": "high",\n  "ticket_id": "TICKET_123456",\n  "customer_name": "Customer Name",\n 

In [14]:
t0 = time.time()
b2 = generate(base, dev, system=RULES_SYSTEM, desc="baseline-2")
r_b2 = report(score(dev, b2), "B2: base model, FULL RULES in prompt (reference only, not allowed at test time)")
print("  minutes:", round((time.time()-t0)/60, 1))
json.dump({"b1": r_b1, "b2": r_b2}, open(os.path.join(OUT_DIR, "baselines.json"), "w"), indent=1)
free(base)

  baseline-2 8/200  5s
  baseline-2 48/200  30s
  baseline-2 88/200  54s
  baseline-2 128/200  85s
  baseline-2 168/200  119s

=== B2: base model, FULL RULES in prompt (reference only, not allowed at test time) ===
  n=200  json_valid=97.5%  exact_match=0.0%  mean_field_acc=42.9%
  per-field: category=57%  priority=0%  amount_paise=82%  txn_date=47%  txn_id=46%  channel=68%  language=16%  needs_human=28%
  minutes: 2.3


### C.1 Output key order: a free scratchpad inside the required JSON

`score.py` builds its hits with `pred[k]` lookups and checks `set(pred) == set(KEYS)`. Nothing in it
is sensitive to the **order** the keys are emitted in — verified in the cell below against the real
scorer. That matters, because the schema's own order works against an autoregressive model:

```
schema order:  category, priority, amount_paise, ...
                         ^^^^^^^^  ^^^^^^^^^^^^
                         committed BEFORE the amount is written
```

`priority` is a function of the amount (>= Rs 50,000 -> P1, >= Rs 5,000 -> P2), and `needs_human` is
a function of `priority`. Generating left to right in schema order, the model has to extract the
amount, compare it to two thresholds and commit to a priority **without writing the amount down** —
it gets no benefit from its own extraction. Reordering the training targets to

```
category -> amount_paise -> priority -> needs_human -> txn_date -> txn_id -> channel -> language
```

means that when the model emits `priority` it can attend to the amount digits it just produced, and
`needs_human` can attend to the priority it just produced. This is the chain-of-thought mechanism
obtained for free inside a fixed 8-key schema: no extra tokens, no rules in the prompt, no
post-processing, no constrained decoding. Only the order of the training targets changes.

**Stated plainly because it should be raised rather than discovered:** this relies on the scorer
being order-insensitive. The model still does all of the work and the output is still exactly one
JSON object with exactly the eight required keys, so it is within the rules as written — but it is a
deliberate exploitation of a property of the scorer and is reported as such.

Tested as experiment **E5** in Part D. Prediction, recorded before the run: `priority` and `needs_human` improve, `amount_paise` does not
move, and the gain concentrates on rows near the two rupee thresholds.

In [15]:
DEP_ORDER = ["category","amount_paise","priority","needs_human","txn_date","txn_id","channel","language"]
assert set(DEP_ORDER) == set(KEYS) and len(DEP_ORDER) == 8

# Verify against the REAL scorer that key order cannot change a score.
_probe_gold = dev[:25]
_probe_pred = [{"id": g["id"],
                "output": json.dumps({k: json.loads(g["messages"][-1]["content"])[k] for k in DEP_ORDER})}
               for g in _probe_gold]
_r = score(_probe_gold, _probe_pred)["all"]
print("gold labels re-emitted in DEP_ORDER ->",
      f"exact_match={_r['exact_match']:.0%} mean_field_acc={_r['mean_field_acc']:.0%}")
assert _r["exact_match"] == 1.0, "key order DOES affect scoring - abandon this experiment"
print("confirmed: key order is not scored. Reordering targets is therefore free.\n")
print("schema order:", KEYS)
print("dep order   :", DEP_ORDER)

def reorder_labels(rows, order):
    """Re-serialise each assistant label with the given key order. Values are untouched."""
    if order == "schema": return rows
    keys = DEP_ORDER if order == "dep" else order
    out = []
    for r in rows:
        o = json.loads(r["messages"][-1]["content"])
        r2 = json.loads(json.dumps(r))
        r2["messages"][-1]["content"] = json.dumps(OrderedDict((k, o[k]) for k in keys), ensure_ascii=False)
        out.append(r2)
    return out

print("\nexample retargeted label:")
print("  schema:", train_clean[0]["messages"][-1]["content"])
print("  dep   :", reorder_labels(train_clean[:1], "dep")[0]["messages"][-1]["content"])

gold labels re-emitted in DEP_ORDER -> exact_match=100% mean_field_acc=100%
confirmed: key order is not scored. Reordering targets is therefore free.

schema order: ['category', 'priority', 'amount_paise', 'txn_date', 'txn_id', 'channel', 'language', 'needs_human']
dep order   : ['category', 'amount_paise', 'priority', 'needs_human', 'txn_date', 'txn_id', 'channel', 'language']

example retargeted label:
  schema: {"category": "payment_failed", "priority": "P3", "amount_paise": 470000, "txn_date": "2026-07-25", "txn_id": "NP1599046332", "channel": "netbanking", "language": "en", "needs_human": false}
  dep   : {"category": "payment_failed", "amount_paise": 470000, "priority": "P3", "needs_human": false, "txn_date": "2026-07-25", "txn_id": "NP1599046332", "channel": "netbanking", "language": "en"}


## Part C — The fine-tuning pipeline

Four decisions the brief asks me to justify.

**Chat template.** Qwen2.5 is ChatML with a real `system` role, so the fixed system prompt passes
through untouched and no fallback is needed. Training sequences are built with the *same*
`apply_chat_template` call used at inference, so there is no train/serve skew.

**Loss on the right tokens.** I tokenize and mask by hand rather than relying on a library matching
a `response_template` string. Everything up to and including `<|im_start|>assistant\n` is set to
`-100`; loss falls only on the label JSON and the closing `<|im_end|>`. The cell below **asserts**
that the unmasked span decodes back to exactly the label, so a silent masking bug fails loudly
instead of quietly producing a mediocre model. Without this the model spends most of its gradient
learning to reproduce support tickets.

**Sequence length.** Chosen from the measured token distribution, not a round number: long enough
that no label can be right-truncated (the label sits at the end of the sequence, so truncation
corrupts the *target*). The bimodal tail is handled with `group_by_length` rather than by shortening.

**The rest.** Effective batch 16 (4 x grad-accum 4) to fit T4 memory while keeping the step count
reasonable at ~1.3k rows. LR 2e-4 with cosine decay and 3% warmup, the standard LoRA starting point,
tested in experiment 3. `r=16, alpha=32` on all attention and MLP projections: 18.5M trainable
params, within the 50M cap. Note that **r=64 on all-linear would be 73.9M and breach the cap** —
the rank sweep is therefore r=16 vs r=32 (36.9M), not r=64.

In [16]:
from torch.utils.data import Dataset as TorchDataset
from transformers import Trainer, TrainingArguments
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

ASSISTANT_HEADER = "<|im_start|>assistant\n"

def build_example(row, max_len):
    """Tokenize one row and mask everything that is not the assistant turn."""
    msgs = row["messages"]
    prompt = tok.apply_chat_template(msgs[:2], tokenize=False, add_generation_prompt=True)
    full   = tok.apply_chat_template(msgs,     tokenize=False, add_generation_prompt=False)
    assert full.startswith(prompt), "chat template is not prompt-prefixed; masking assumption broken"
    p_ids = tok(prompt, add_special_tokens=False)["input_ids"]
    f_ids = tok(full,   add_special_tokens=False)["input_ids"]
    labels = [-100] * len(p_ids) + f_ids[len(p_ids):]
    return {"input_ids": f_ids[:max_len], "labels": labels[:max_len]}

class TriageDataset(TorchDataset):
    def __init__(self, rows, max_len):
        self.ex = [build_example(r, max_len) for r in rows]
        self.lengths = [len(e["input_ids"]) for e in self.ex]
    def __len__(self): return len(self.ex)
    def __getitem__(self, i): return self.ex[i]

def collate(batch):
    L = max(len(b["input_ids"]) for b in batch)
    pad = tok.pad_token_id
    return {
        "input_ids":      torch.tensor([b["input_ids"] + [pad]*(L-len(b["input_ids"])) for b in batch]),
        "attention_mask": torch.tensor([[1]*len(b["input_ids"]) + [0]*(L-len(b["input_ids"])) for b in batch]),
        "labels":         torch.tensor([b["labels"]    + [-100]*(L-len(b["labels"]))    for b in batch]),
    }

# ---- PROOF that the loss mask is right. This is the assertion that matters most in Part C. ----
_probe = build_example(train_clean[0], MAX_SEQ_LEN)
_sup = [i for i, l in enumerate(_probe["labels"]) if l != -100]
_decoded = tok.decode([_probe["input_ids"][i] for i in _sup])
_expected = train_clean[0]["messages"][-1]["content"]
print("supervised token span:", _sup[0], "..", _sup[-1], f"({len(_sup)} of {len(_probe['input_ids'])} tokens)")
print("decoded supervised span:\n ", repr(_decoded))
print("expected label:\n ", repr(_expected))
assert _decoded.replace("<|im_end|>", "").strip() == _expected.strip(), "LOSS MASK IS WRONG"
masked_txt = tok.decode([t for t, l in zip(_probe["input_ids"], _probe["labels"]) if l == -100])
assert masked_txt.endswith(ASSISTANT_HEADER), "mask does not end exactly at the assistant header"
print(f"\nMASK VERIFIED: loss falls on {len(_sup)}/{len(_probe['input_ids'])} tokens "
      f"({len(_sup)/len(_probe['input_ids']):.0%}) = the label only.")
print("Everything before and including '<|im_start|>assistant\\n' is -100.")

supervised token span: 133 .. 214 (82 of 215 tokens)
decoded supervised span:
  '{"category": "payment_failed", "priority": "P3", "amount_paise": 470000, "txn_date": "2026-07-25", "txn_id": "NP1599046332", "channel": "netbanking", "language": "en", "needs_human": false}<|im_end|>\n'
expected label:
  '{"category": "payment_failed", "priority": "P3", "amount_paise": 470000, "txn_date": "2026-07-25", "txn_id": "NP1599046332", "channel": "netbanking", "language": "en", "needs_human": false}'

MASK VERIFIED: loss falls on 82/215 tokens (38%) = the label only.
Everything before and including '<|im_start|>assistant\n' is -100.


In [17]:
def lora_cfg(r=16, alpha=None, dropout=0.05):
    return LoraConfig(
        r=r, lora_alpha=alpha if alpha is not None else 2*r, lora_dropout=dropout,
        bias="none", task_type="CAUSAL_LM",
        target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"])

def train_run(name, rows, four_bit=False, r=16, lr=2e-4, epochs=3,
              bs=2, accum=8, max_len=None, save=False, seed=SEED, order="schema"):
    """One controlled run. Returns metrics + the dev predictions.
    `order` selects the key order of the training targets: "schema" or "dep" (see D.5)."""
    max_len = max_len or MAX_SEQ_LEN
    rows = reorder_labels(rows, order)
    free(); gc.collect(); torch.cuda.empty_cache(); torch.cuda.reset_peak_memory_stats()
    t0 = time.time()
    model = load_base(four_bit=four_bit)
    if four_bit:
        model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    model.config.use_cache = False
    model = get_peft_model(model, lora_cfg(r=r))
    # fp16 AMP keeps fp32 master weights for whatever it optimises. If the LoRA params inherit the
    # base model's float16 the grad scaler dies with "Attempting to unscale FP16 gradients" partway
    # into the run. peft>=0.12 is supposed to handle this via autocast_adapter_dtype, but this is a
    # hard crash on a metered GPU, so it is forced and asserted rather than assumed.
    # Gradient checkpointing is needed on BOTH arms, not just 4-bit: MAX_SEQ_LEN is ~2.2k to avoid
    # truncating labels, and group_by_length packs the long quoted-email rows into the SAME batch,
    # so the worst-case batch is the one that decides whether this fits a 15 GB T4. Without it,
    # bs=2 x 2240 tokens in fp16 OOMs. For a non-quantized model PEFT needs input grads enabled
    # explicitly, or checkpointing yields "element 0 of tensors does not require grad".
    model.gradient_checkpointing_enable(gradient_checkpointing_kwargs={"use_reentrant": False})
    model.enable_input_require_grads()
    for _n, _p in model.named_parameters():
        if _p.requires_grad and _p.dtype != torch.float32:
            _p.data = _p.data.to(torch.float32)
    _bad = [n for n, q in model.named_parameters() if q.requires_grad and q.dtype != torch.float32]
    assert not _bad, f"trainable params not fp32: {_bad[:3]}"
    trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
    assert trainable <= 50_000_000, f"{trainable/1e6:.1f}M trainable exceeds the 50M cap"
    print(f"[{name}] trainable {trainable/1e6:.2f}M  ({trainable/sum(p.numel() for p in model.parameters()):.2%} of total)")

    ds = TriageDataset(rows, max_len)
    args = TrainingArguments(
        output_dir=f"/content/run_{name}", num_train_epochs=epochs,
        per_device_train_batch_size=bs, gradient_accumulation_steps=accum,
        learning_rate=lr, lr_scheduler_type="cosine", warmup_ratio=0.03,
        logging_steps=25, save_strategy="no", report_to=[],
        fp16=True, bf16=False,                  # T4 / Turing
        optim="paged_adamw_8bit" if four_bit else "adamw_torch",
        gradient_checkpointing=True, gradient_checkpointing_kwargs={"use_reentrant": False},
        group_by_length=True,
        seed=seed, max_grad_norm=1.0, remove_unused_columns=False,
    )
    tr = Trainer(model=model, args=args, train_dataset=ds, data_collator=collate)
    hist = tr.train()
    mins = (time.time() - t0) / 60
    peak = torch.cuda.max_memory_allocated() / 1e9

    # checkpointing only costs time at inference; turn it back off and re-enable the kv cache
    try: model.gradient_checkpointing_disable()
    except Exception: pass
    model.config.use_cache = True
    free()
    preds = generate(model, dev, system=FIXED_SYSTEM, desc=f"[{name}] dev")
    res = score(dev, preds)
    m = {"name": name, "n_train": len(rows), "four_bit": four_bit, "r": r, "lr": lr,
         "epochs": epochs, "max_len": max_len, "order": order, "trainable_M": round(trainable/1e6, 2),
         "exact_match": res["all"]["exact_match"], "mean_field_acc": res["all"]["mean_field_acc"],
         "json_valid": res["all"]["json_valid"], "peak_gpu_gb": round(peak, 2),
         "train_minutes": round(mins, 1), "final_loss": round(hist.training_loss, 4),
         "fields": res["all"]["fields"]}
    print(f"[{name}] exact={m['exact_match']:.1%} field={m['mean_field_acc']:.1%} "
          f"peak={m['peak_gpu_gb']}GB mins={m['train_minutes']}")
    if save:
        adir = os.path.join(OUT_DIR, "adapter")
        if os.path.isdir(adir): shutil.rmtree(adir)
        model.save_pretrained(adir); tok.save_pretrained(adir)
        print("adapter saved ->", adir, os.listdir(adir))
    out_model = model if save else None
    if not save: free(model)
    return m, preds, out_model

## Part D — Controlled experiments

Each row changes **one** thing against the E0 reference (cleaned data, 16-bit LoRA, r=16, lr 2e-4,
3 epochs).

| # | Variable | Arms | Hypothesis |
|---|---|---|---|
| 1 | training data | raw 1445 vs cleaned 1350 | The legacy `low/medium/high` values and the 62 priority violations teach the model contradictory priority rules, so `priority` and `needs_human` should move most. |
| 2 | precision | 16-bit LoRA vs 4-bit QLoRA | QLoRA should cut peak memory substantially for a small accuracy cost. On a 1.5B model 16-bit already fits a T4, so QLoRA buys headroom rather than feasibility. |
| 3 | epochs | 2 vs 3 vs 5 | 1.3k rows with a rigid output format is a small, highly templated task; I expect fast saturation and possible degradation by 5. |
| 4 | **my own:** category-balanced oversampling | natural vs fraud/offers/other oversampled | `fraud` is only ~6% of rows but it *overrides every other category* and forces `priority=P1` and `needs_human=true`. One missed fraud therefore costs three fields at once, so rare-category errors are mispriced by a flat sampler. |

In [18]:
def oversample_rare(rows, targets=("fraud","offers","other"), factor=2):
    """Experiment 4. Duplicate rows whose category is rare but high-leverage.

    Rationale: a missed `fraud` cascades into priority and needs_human, so its true cost is ~3x a
    normal field error. Duplication (rather than loss weighting) keeps the change to a single
    variable -- the sampling distribution -- with the optimiser untouched."""
    out = list(rows)
    for r in rows:
        if json.loads(r["messages"][-1]["content"])["category"] in targets:
            out.extend([r] * (factor - 1))
    random.Random(SEED).shuffle(out)
    return out

print("natural category mix:")
nat = Counter(json.loads(r["messages"][-1]["content"])["category"] for r in train_clean)
for k, v in nat.most_common(): print(f"   {k:16} {v:4d}  {v/len(train_clean):5.1%}")
over = oversample_rare(train_clean)
print(f"\noversampled: {len(train_clean)} -> {len(over)} rows")
ov = Counter(json.loads(r["messages"][-1]["content"])["category"] for r in over)
for k, v in ov.most_common(): print(f"   {k:16} {v:4d}  {v/len(over):5.1%}")

natural category mix:
   payment_failed    393  29.1%
   refund            298  22.1%
   account_access    174  12.9%
   kyc               162  12.0%
   offers            142  10.5%
   other              97   7.2%
   fraud              84   6.2%

oversampled: 1350 -> 1673 rows
   payment_failed    393  23.5%
   refund            298  17.8%
   offers            284  17.0%
   other             194  11.6%
   account_access    174  10.4%
   fraud             168  10.0%
   kyc               162   9.7%


### D.6 Counterfactual minimal pairs

The spec's whole decision surface rests on a handful of triggers, and the cleaned data teaches them
thinly:

| Trigger | Rows that exercise it | Share |
|---|---:|---:|
| Rs 5,000 P2 boundary (+-25%) | 61 | 4.5% |
| Rs 50,000 P1 boundary (+-25%) | 45 | 3.3% |
| legal threat — the only P1 trigger outside fraud/amount | 87 | 6.4% |
| repeat contact — the only `needs_human` trigger outside P1 | 125 | 9.3% |
| quoted `>` email history | 38 | 2.8% (vs **11 of 400 test rows**) |

Because the rule engine is provably exact on dev, it can act as a **label oracle**: mutate one
trigger in a real ticket and recompute the label deterministically. That produces *minimal pairs* —
two rows identical except for the feature that flips the label — which isolate the decision boundary
in the gradient far more efficiently than more random examples. This uses only `train.jsonl` and
`SCHEMA.md`, and no other model, so it is within the data rules.

Four generators, all paired with their unmutated original:

1. **Threshold straddling** — rewrite the money figure to just under and just over a boundary.
   Restricted to rows where the labelled amount appears exactly once in an explicit-currency form,
   so the rewrite cannot leave a second stale figure behind.
2. **Legal-threat injection** — append one clause; `priority` -> P1 and `needs_human` -> true.
3. **Repeat-contact injection** — append one phrase; `needs_human` flips.
4. **Quoted-email splicing** — graft real `>` blocks onto short tickets; the label is **unchanged**,
   which is the whole point: it teaches that quoted history must be ignored.

In [19]:
MONEY_EXPLICIT = re.compile(r"(₹|Rs\.?\s*|INR\s*)(\d[\d,]*(?:\.\d+)?)|(\d[\d,]*(?:\.\d+)?)(\s*(?:rupees|rupaye))", re.I)
LEGAL_CLAUSES = ["I will complain to the RBI ombudsman.", "My lawyer will send a legal notice.",
                 "I am taking this to consumer court.", "I will be forced to take legal action."]
REPEAT_CLAUSES = ["This is the third time I am contacting you.", "I keep raising this again and again.",
                  "This is my second complaint on the same issue.", "Pehle bhi complaint kar chuka hoon."]

def _single_money_span(body, paise):
    """The one explicit-currency mention equal to `paise`, or None if 0 or >1 match."""
    hits = []
    for m in MONEY_EXPLICIT.finditer(body):
        num = m.group(2) or m.group(3)
        if num is None: continue
        try: v = round(float(num.replace(",", "")) * 100)
        except ValueError: continue
        if v == paise: hits.append(m)
    return hits[0] if len(hits) == 1 else None

def _relabel(row, body_for_rules, overrides):
    """Rebuild a label: apply overrides, then recompute the derived fields by rule."""
    o = json.loads(row["messages"][-1]["content"]); o.update(overrides)
    o["priority"] = priority_of(o["category"], o["amount_paise"], body_for_rules)
    o["needs_human"] = needs_human_of(o["priority"], body_for_rules)
    return o

def _emit(row, new_user, overrides, tag, n):
    r2 = json.loads(json.dumps(row))
    r2["messages"][1]["content"] = new_user
    r2["id"] = f"{row['id']}-{tag}{n}"
    o = _relabel(row, body_of(new_user), overrides)
    r2["messages"][-1]["content"] = json.dumps(OrderedDict((k, o[k]) for k in KEYS), ensure_ascii=False)
    return r2

def make_counterfactuals(rows, seed=SEED, cap_per_kind=180):
    rng = random.Random(seed); out = []; made = Counter()
    THRESHOLDS = [500_000, 5_000_000]            # Rs 5,000 and Rs 50,000 in paise
    for r in rows:
        o = json.loads(r["messages"][-1]["content"])
        user = r["messages"][1]["content"]; body = body_of(user)
        # (1) threshold straddling
        if (o["category"] in ("payment_failed","refund") and isinstance(o["amount_paise"], int)
                and made["thr"] < cap_per_kind):
            span = _single_money_span(body, o["amount_paise"])
            if span is not None:
                T = rng.choice(THRESHOLDS)
                for k, delta in enumerate((-1, +1)):
                    rupees = T // 100 + delta * rng.randint(20, 400)
                    new_num = f"{rupees:,}"
                    s, e = span.span()
                    seg = body[s:e]
                    old_num = span.group(2) or span.group(3)
                    new_body = body[:s] + seg.replace(old_num, new_num, 1) + body[e:]
                    new_user = user.replace(body, new_body, 1)
                    if new_user == user: continue
                    out.append(_emit(r, new_user, {"amount_paise": rupees * 100}, "thr", k)); made["thr"] += 1
        # (2) legal-threat injection
        if not LEGAL.search(body) and made["legal"] < cap_per_kind:
            new_body = body.rstrip() + " " + rng.choice(LEGAL_CLAUSES)
            out.append(_emit(r, user.replace(body, new_body, 1), {}, "legal", 0)); made["legal"] += 1
        # (3) repeat-contact injection
        if not REPEAT.search(body) and made["repeat"] < cap_per_kind:
            new_body = body.rstrip() + " " + rng.choice(REPEAT_CLAUSES)
            out.append(_emit(r, user.replace(body, new_body, 1), {}, "rep", 0)); made["repeat"] += 1
    # (4) quoted-email splicing: real quoted blocks grafted onto short tickets, label unchanged
    donors = []
    for r in rows:
        q = [l for l in r["messages"][1]["content"].split("\n") if l.lstrip().startswith(">")]
        if len(q) >= 2: donors.append("\n".join(q))
    if donors:
        shorts = [r for r in rows if ">" not in r["messages"][1]["content"]]
        rng.shuffle(shorts)
        for i, r in enumerate(shorts[:cap_per_kind]):
            blk = "\n\n".join(rng.sample(donors, min(len(donors), rng.randint(1, 3))))
            r2 = json.loads(json.dumps(r))
            r2["messages"][1]["content"] = r["messages"][1]["content"].rstrip() + "\n\n" + blk
            r2["id"] = f"{r['id']}-quote"
            out.append(r2); made["quote"] += 1   # label deliberately unchanged
    rng.shuffle(out)
    return out, made

cf, cf_made = make_counterfactuals(train_clean)
print("counterfactual rows generated:", dict(cf_made), "total", len(cf))

# Every generated row must satisfy the same validator dev satisfies.
cf_flags = validate(cf)
print("validator on generated rows:", {k: len(v) for k, v in cf_flags.items()} or "ZERO - all generated labels are spec-consistent")
assert not cf_flags, "counterfactual generator produced a spec-inconsistent label"

train_cf = train_clean + cf
print(f"\naugmented training set: {len(train_clean)} + {len(cf)} = {len(train_cf)} rows")
print("\nexample threshold minimal pair:")
for r in cf:
    if r["id"].endswith("thr0"):
        base = next(x for x in train_clean if x["id"] == r["id"].rsplit("-", 1)[0])
        print("  ORIGINAL:", body_of(base["messages"][1]["content"])[:110])
        print("        ->", base["messages"][-1]["content"])
        print("  MUTATED :", body_of(r["messages"][1]["content"])[:110])
        print("        ->", r["messages"][-1]["content"])
        break
nb_thr = sum(1 for r in train_cf
             if (lambda a: a is not None and (450_000 <= a <= 550_000 or 4_500_000 <= a <= 5_500_000))
             (json.loads(r["messages"][-1]["content"])["amount_paise"]))
print(f"\nrows within +-10% of a rupee threshold: 106 natural -> {nb_thr} after augmentation")

counterfactual rows generated: {'legal': 180, 'repeat': 180, 'thr': 180, 'quote': 180} total 720
validator on generated rows: ZERO - all generated labels are spec-consistent

augmented training set: 1350 + 720 = 2070 rows

example threshold minimal pair:
  ORIGINAL: Hey, I have a duplicate payment showing. This happened on 08-08-26. It was from the NimbusPay wallet. Transact
        -> {"category": "refund", "priority": "P2", "amount_paise": 1938900, "txn_date": "2026-08-08", "txn_id": "NP2759013242", "channel": "wallet", "language": "en", "needs_human": false}
  MUTATED : Hey, I have a duplicate payment showing. This happened on 08-08-26. It was from the NimbusPay wallet. Transact
        -> {"category": "refund", "priority": "P3", "amount_paise": 489200, "txn_date": "2026-08-08", "txn_id": "NP2759013242", "channel": "wallet", "language": "en", "needs_human": false}

rows within +-10% of a rupee threshold: 106 natural -> 264 after augmentation


In [20]:
EXP_PATH = os.path.join(OUT_DIR, "experiments_results.json")

EXPERIMENTS = [
    ("E0_cleaned_baseline", dict(rows_key="clean", four_bit=False, r=16, lr=2e-4, epochs=3)),
    ("E1a_raw_data",        dict(rows_key="raw",   four_bit=False, r=16, lr=2e-4, epochs=3)),
    ("E2a_qlora_4bit",      dict(rows_key="clean", four_bit=True,  r=16, lr=2e-4, epochs=3)),
    ("E3a_epochs_2",        dict(rows_key="clean", four_bit=False, r=16, lr=2e-4, epochs=2)),
    ("E3b_epochs_5",        dict(rows_key="clean", four_bit=False, r=16, lr=2e-4, epochs=5)),
    ("E3c_rank_32",         dict(rows_key="clean", four_bit=False, r=32, lr=2e-4, epochs=3)),
    ("E4_oversample_rare",  dict(rows_key="over",  four_bit=False, r=16, lr=2e-4, epochs=3)),
    # --- the two ideas from C.1 / D.6, each one variable away from E0 ---
    ("E5_key_order_dep",    dict(rows_key="clean", four_bit=False, r=16, lr=2e-4, epochs=3, order="dep")),
    ("E6_counterfactual",   dict(rows_key="cf",    four_bit=False, r=16, lr=2e-4, epochs=3)),
    # --- combination run: changes TWO things at once, so it is not a controlled comparison and is
    # --- reported separately. Included only to see whether the two gains stack.
    ("E7_combo_cf_plus_dep", dict(rows_key="cf",   four_bit=False, r=16, lr=2e-4, epochs=3, order="dep")),
]

def rows_for(key):
    if key == "clean": return train_clean
    if key == "over":  return over
    if key == "cf":    return train_cf
    if key == "raw":
        # raw arm: only the rows whose label is parseable at all, since an unparseable label cannot
        # be made into a training target. That is the honest "no cleaning" condition -- 1420 rows.
        keep = []
        for r in train_raw:
            try: json.loads(r["messages"][-1]["content"])
            except Exception: continue
            keep.append(r)
        return keep
    raise KeyError(key)

if RUN_EXPERIMENTS:
    results = []
    for name, cfg in EXPERIMENTS:
        k = cfg.pop("rows_key")
        m, _, _ = train_run(name, rows_for(k), **cfg)
        m["rows_key"] = k; results.append(m)
        json.dump(results, open(EXP_PATH, "w"), indent=1)   # checkpoint after every run
else:
    results = json.load(open(EXP_PATH)) if os.path.isfile(EXP_PATH) else []
    print("RUN_EXPERIMENTS=False -> loaded", len(results), "saved runs from", EXP_PATH)
    if not results:
        print("\nNo saved results yet. Set RUN_EXPERIMENTS=True and rerun this notebook once")
        print("to generate experiments_results.json, then set it back to False.")

RUN_EXPERIMENTS=False -> loaded 0 saved runs from /content/out/experiments_results.json

No saved results yet. Set RUN_EXPERIMENTS=True and rerun this notebook once
to generate experiments_results.json, then set it back to False.


In [21]:
if results:
    hdr = (f"{'run':24} {'data':6} {'keys':6} {'bits':5} {'r':>3} {'lr':>7} {'ep':>3} "
           f"{'exact':>7} {'field':>7} {'GPU GB':>7} {'mins':>6}")
    print(hdr); print("-"*len(hdr))
    for m in results:
        print(f"{m['name']:24} {m.get('rows_key','clean'):6} {m.get('order','schema'):6} "
              f"{'4bit' if m['four_bit'] else '16bit':5} "
              f"{m['r']:3d} {m['lr']:7.0e} {m['epochs']:3d} {m['exact_match']:7.1%} "
              f"{m['mean_field_acc']:7.1%} {m['peak_gpu_gb']:7.2f} {m['train_minutes']:6.1f}")
    print("\nper-field accuracy by run")
    print(f"{'run':24} " + " ".join(f"{k[:9]:>9}" for k in KEYS))
    for m in results:
        print(f"{m['name']:24} " + " ".join(f"{m['fields'][k]:9.1%}" for k in KEYS))
    best = max(results, key=lambda m: m["exact_match"])
    print(f"\nbest: {best['name']}  exact={best['exact_match']:.1%}  field={best['mean_field_acc']:.1%}")

## Final model

Trained with the best configuration from Part D on the cleaned data, then saved as a standard PEFT
adapter (`adapter_config.json` + `adapter_model.safetensors`).

In [22]:
BEST = dict(four_bit=False, r=16, lr=2e-4, epochs=3, order="schema")   # <-- set from the Part D table
BEST_ROWS = "clean"
if results:
    b = max(results, key=lambda m: m["exact_match"])
    BEST = dict(four_bit=b["four_bit"], r=b["r"], lr=b["lr"], epochs=b["epochs"],
                order=b.get("order", "schema"))
    BEST_ROWS = b.get("rows_key", "clean")
    print("adopting best config from experiments:", b["name"], BEST)
else:
    print("no experiment file; using the documented default config:", BEST)

final_m, final_dev_preds, final_model = train_run(
    "FINAL", rows_for(BEST_ROWS), save=True, **BEST)
res_final = score(dev, final_dev_preds)
_ = report(res_final, "FINAL model on dev")
print("\nby category:")
for k in sorted(res_final):
    if k.startswith("category="):
        r = res_final[k]; print(f"  {k:28} n={r['n']:3d} exact={r['exact_match']:6.1%} field={r['mean_field_acc']:6.1%}")
json.dump(final_m, open(os.path.join(OUT_DIR, "final_metrics.json"), "w"), indent=1)
with open(os.path.join(OUT_DIR, "dev_predictions.jsonl"), "w", encoding="utf-8") as f:
    for p in final_dev_preds: f.write(json.dumps(p, ensure_ascii=False) + "\n")

no experiment file; using the documented default config: {'four_bit': False, 'r': 16, 'lr': 0.0002, 'epochs': 3, 'order': 'schema'}
  base loaded, logits finite, 3.10 GB
[FINAL] trainable 18.46M  (1.18% of total)


The model is already on multiple devices. Skipping the move to device specified in `args`.


Step,Training Loss
25,0.490600
50,0.051300
75,0.037600
100,0.028100
125,0.017700
150,0.012200
175,0.008000
200,0.004600
225,0.003600
250,0.003000


  [FINAL] dev 8/200  7s
  [FINAL] dev 48/200  44s
  [FINAL] dev 88/200  81s
  [FINAL] dev 128/200  122s
  [FINAL] dev 168/200  166s
[FINAL] exact=90.0% field=97.9% peak=11.82GB mins=19.9
adapter saved -> /content/out/adapter ['adapter_model.safetensors', 'tokenizer_config.json', 'added_tokens.json', 'chat_template.jinja', 'vocab.json', 'special_tokens_map.json', 'tokenizer.json', 'adapter_config.json', 'merges.txt', 'README.md']

=== FINAL model on dev ===
  n=200  json_valid=100.0%  exact_match=90.0%  mean_field_acc=97.9%
  per-field: category=98%  priority=92%  amount_paise=100%  txn_date=99%  txn_id=100%  channel=100%  language=100%  needs_human=94%

by category:
  category=account_access      n= 19 exact=100.0% field=100.0%
  category=fraud               n= 15 exact= 80.0% field= 94.2%
  category=kyc                 n= 24 exact=100.0% field=100.0%
  category=offers              n= 18 exact= 94.4% field= 99.3%
  category=other               n= 16 exact=100.0% field=100.0%
  category

## Part E — Error analysis

In [23]:
a = res_final["all"]
print("weakest fields (final model, dev):")
for k, v in sorted(a["fields"].items(), key=lambda kv: kv[1]):
    print(f"   {k:14} {v:6.1%}")
print("\nweakest categories:")
cats = [(k.split('=')[1], res_final[k]) for k in res_final if k.startswith("category=")]
for c, r in sorted(cats, key=lambda x: x[1]["exact_match"]):
    print(f"   {c:16} n={r['n']:3d} exact={r['exact_match']:6.1%} field={r['mean_field_acc']:6.1%}")

pmap = {p["id"]: p["output"] for p in final_dev_preds}
failures = []
for g in dev:
    gold = json.loads(g["messages"][-1]["content"])
    pred = _parse(pmap.get(g["id"]))
    if pred is None:
        failures.append((g["id"], "INVALID_JSON", gold, pmap.get(g["id"]), g)); continue
    diffs = {k: (gold[k], pred.get(k, "<missing>")) for k in KEYS if not _same(pred.get(k), gold[k])}
    if diffs: failures.append((g["id"], diffs, gold, pred, g))
print(f"\n{len(failures)}/{len(dev)} dev rows have at least one wrong field")
wrong_field = Counter()
for _, d, *_ in failures:
    if d == "INVALID_JSON": wrong_field["<invalid json>"] += 1
    else:
        for k in d: wrong_field[k] += 1
print("field error counts:", dict(wrong_field.most_common()))

weakest fields (final model, dev):
   priority        91.5%
   needs_human     94.5%
   category        98.0%
   txn_date        99.0%
   amount_paise   100.0%
   txn_id         100.0%
   channel        100.0%
   language       100.0%

weakest categories:
   fraud            n= 15 exact= 80.0% field= 94.2%
   payment_failed   n= 53 exact= 81.1% field= 96.5%
   refund           n= 55 exact= 89.1% field= 97.5%
   offers           n= 18 exact= 94.4% field= 99.3%
   other            n= 16 exact=100.0% field=100.0%
   kyc              n= 24 exact=100.0% field=100.0%
   account_access   n= 19 exact=100.0% field=100.0%

20/200 dev rows have at least one wrong field
field error counts: {'priority': 17, 'needs_human': 11, 'category': 4, 'txn_date': 2}


### E.1 Automated error attribution

Counting wrong fields overstates the number of independent mistakes, because the spec is a
dependency graph: `needs_human` is a function of `priority`, and `priority` is a function of
`category` and `amount_paise`. A single amount error shows up as three wrong fields.

So each failure is classified by **root cause**, and any field whose value is correct *given the
model's own upstream prediction* is labelled a **cascade** rather than counted again. That is the
difference between "priority was wrong 20 times" and "priority was wrong 20 times, 13 of which were
the ladder applied correctly to an amount the model had already misread."

In [24]:
QUOTED_DATE = re.compile(r"(\d{1,2})\s+(Jan|Feb|Mar|Apr|May|Jun|Jul|Aug|Sep|Oct|Nov|Dec)\s+(\d{4})", re.I)
MONTHS = {m: i+1 for i, m in enumerate(["jan","feb","mar","apr","may","jun","jul","aug","sep","oct","nov","dec"])}

def quoted_dates(user):
    out = set()
    for l in user.split("\n"):
        if not l.lstrip().startswith(">"): continue
        for m in QUOTED_DATE.finditer(l):
            try: out.add(datetime.date(int(m.group(3)), MONTHS[m.group(2).lower()], int(m.group(1))).isoformat())
            except ValueError: pass
    return out

def attribute(gold, pred, user):
    """Return {field: cause}. Causes prefixed CASCADE are downstream of another error."""
    if pred is None: return {"<output>": "not parseable as JSON"}
    if set(pred) != set(KEYS): return {"<output>": f"wrong key set: {sorted(set(pred)^set(KEYS))}"}
    body, recv = body_of(user), recv_of(user)
    wrong = {k for k in KEYS if not _same(pred.get(k), gold[k])}
    c = {}

    if "category" in wrong:
        c["category"] = f"confused {gold['category']} -> {pred['category']}"

    if "amount_paise" in wrong:
        g, p = gold["amount_paise"], pred["amount_paise"]
        if isinstance(p, int) and isinstance(g, int) and (p == g * 100 or g == p * 100):
            c["amount_paise"] = "unit error: rupees/paise confusion (100x)"
        elif p is None and g is not None:
            c["amount_paise"] = "missed the amount (emitted null)"
        elif g is None and p is not None:
            c["amount_paise"] = ("invented an amount; the ticket states none"
                                 if gold["category"] not in NO_TXN else
                                 f"emitted a txn field for category={gold['category']} (must be null)")
        elif isinstance(p, int) and p in amount_candidates(body):
            c["amount_paise"] = "picked the wrong figure in the ticket (distractor: balance/limit/fee)"
        else:
            c["amount_paise"] = "wrong magnitude (k/lakh conversion or digit error)"

    if "priority" in wrong:
        own = priority_of(pred["category"],
                          pred["amount_paise"] if isinstance(pred["amount_paise"], int) else None,
                          body)
        if own == pred["priority"] and ({"category","amount_paise"} & wrong):
            c["priority"] = "CASCADE: ladder correct for its own (wrong) category/amount"
        elif gold["priority"] == "P1" and LEGAL.search(body) and pred["priority"] != "P1":
            c["priority"] = "missed the legal-threat P1 trigger"
        elif gold["priority"] == "P1" and gold["category"] == "fraud":
            c["priority"] = "missed that fraud forces P1"
        elif isinstance(gold["amount_paise"], int) and (
                abs(gold["amount_paise"] - 500_000) < 150_000 or abs(gold["amount_paise"] - 5_000_000) < 1_500_000):
            c["priority"] = "near a rupee threshold: boundary misapplied"
        else:
            c["priority"] = f"ladder misapplied ({gold['priority']} -> {pred['priority']})"

    if "needs_human" in wrong:
        if "priority" in wrong:
            c["needs_human"] = "CASCADE: consistent with its own wrong priority"
        elif gold["needs_human"] and REPEAT.search(body):
            c["needs_human"] = "missed repeat-contact wording"
        elif not gold["needs_human"] and pred["needs_human"]:
            c["needs_human"] = "spurious escalation (no P1, no repeat contact)"
        else:
            c["needs_human"] = "rule misapplied"

    if "txn_date" in wrong:
        g, p = gold["txn_date"], pred["txn_date"]
        # most-specific diagnosis FIRST: a generic bucket placed earlier swallows the real cause.
        if p is None: c["txn_date"] = "missed the date"
        elif p in quoted_dates(user):
            c["txn_date"] = "took a date from the quoted '>' email history (spec forbids this)"
        elif g is None:
            c["txn_date"] = ("invented a date" if gold["category"] not in NO_TXN
                             else f"emitted a txn field for category={gold['category']} (must be null)")
        else:
            try:
                gd, pd = datetime.date.fromisoformat(g), datetime.date.fromisoformat(p)
                if (gd.month, gd.day) == (pd.month, pd.day) and gd.year != pd.year:
                    c["txn_date"] = f"year wrong ({pd.year} vs {gd.year}): most-recent-year rule"
                else:
                    c["txn_date"] = f"relative-date arithmetic off by {(pd-gd).days:+d} days"
            except ValueError:
                c["txn_date"] = f"malformed date {p!r}"

    if "txn_id" in wrong:
        g, p = gold["txn_id"], pred["txn_id"]
        up = re.sub(r"[^A-Z0-9]", "", (p or "").upper())
        if p is None: c["txn_id"] = "missed the transaction id"
        elif g is None: c["txn_id"] = "emitted an id the ticket does not contain"
        elif up == re.sub(r"[^A-Z0-9]", "", g.upper()): c["txn_id"] = "normalisation (case/spaces/hyphens)"
        elif re.sub(r"\D", "", p) and re.sub(r"\D", "", p) in re.sub(r"[^0-9]", "", user):
            c["txn_id"] = "copied a ticket/bank/order reference instead of the NP id"
        else: c["txn_id"] = "digits wrong (transcription)"

    if "channel" in wrong:
        c["channel"] = ("inferred a channel that is not explicitly stated" if gold["channel"] is None
                        else ("missed the stated channel" if pred["channel"] is None
                              else f"wrong mode ({gold['channel']} -> {pred['channel']})"))

    if "language" in wrong:
        c["language"] = f"{gold['language']} read as {pred['language']} (Hinglish boundary)"

    return c

roots, cascades, by_field = Counter(), Counter(), Counter()
attributed = []
for g in dev:
    gold = json.loads(g["messages"][-1]["content"])
    pred = _parse(pmap.get(g["id"]))
    cs = attribute(gold, pred, g["messages"][1]["content"])
    if not cs: continue
    attributed.append((g["id"], cs, gold, pred, g))
    for f, why in cs.items():
        by_field[f] += 1
        (cascades if why.startswith("CASCADE") else roots)[f"{f}: {why}"] += 1

print(f"{len(attributed)}/{len(dev)} dev rows have at least one error\n")
print("ROOT CAUSES (most common first)")
for k, v in roots.most_common(20): print(f"  {v:4d}  {k}")
print("\nCASCADES (downstream of another error, not independent failures)")
for k, v in cascades.most_common(): print(f"  {v:4d}  {k}")
print(f"\ntotal wrong fields {sum(by_field.values())} = {sum(roots.values())} root + {sum(cascades.values())} cascade")
print("-> independent mistakes are far fewer than the raw per-field error count suggests.")
json.dump({"roots": dict(roots), "cascades": dict(cascades)},
          open(os.path.join(OUT_DIR, "error_attribution.json"), "w"), indent=1)

20/200 dev rows have at least one error

ROOT CAUSES (most common first)
     9  priority: near a rupee threshold: boundary misapplied
     2  priority: ladder misapplied (P2 -> P1)
     2  category: confused fraud -> refund
     2  txn_date: relative-date arithmetic off by +1 days
     2  priority: ladder misapplied (P2 -> P3)
     1  priority: ladder misapplied (P1 -> P3)
     1  category: confused fraud -> account_access
     1  category: confused payment_failed -> fraud

CASCADES (downstream of another error, not independent failures)
    11  needs_human: CASCADE: consistent with its own wrong priority
     3  priority: CASCADE: ladder correct for its own (wrong) category/amount

total wrong fields 34 = 20 root + 14 cascade
-> independent mistakes are far fewer than the raw per-field error count suggests.


In [25]:
# the 14 worked failures, now each with an attributed cause
for i, (rid, cs, gold, pred, g) in enumerate(attributed[:14], 1):
    print("=" * 100)
    print(f"[{i}] {rid}   Received {recv_of(g['messages'][1]['content'])}")
    print(f"    ticket: {body_of(g['messages'][1]['content'])[:240]!r}")
    for f, why in cs.items():
        gv = gold.get(f, "-"); pv = (pred or {}).get(f, "-")
        print(f"    {f:14} gold={gv!r:26} pred={pv!r:26}")
        print(f"    {'':14} WHY: {why}")
print("=" * 100)

[1] dv-00005   Received 2026-08-09
    ticket: '[IVR transcript] Hey, I tried to make a transfer but it failed. Dated Aug 1. Total 49680 rupees. Ref: NP9512428472 Kindly resolve this asap.'
    priority       gold='P2'                       pred='P1'                      
                   WHY: near a rupee threshold: boundary misapplied
    needs_human    gold=False                      pred=True                      
                   WHY: CASCADE: consistent with its own wrong priority
[2] dv-00034   Received 2026-08-22
    ticket: '@NimbusPayCare Shopping ka payment decline ho gaya lekin bank ne paise kaat liye. Rs. 36737 ka transaction tha. Txn id np 94424 25625. Wallet se hua tha.'
    priority       gold='P2'                       pred='P1'                      
                   WHY: near a rupee threshold: boundary misapplied
    needs_human    gold=False                      pred=True                      
                   WHY: CASCADE: consistent with its own wrong prio

### What I would do with one more week

Filled in from the numbers above once the run completes — see `report.md` for the written version.
The structural candidates, in the order I would try them:

1. **Attack the weakest field directly.** `txn_date` relative-date resolution ("parso", "3 days
   ago", bare "25 Jun" needing the most-recent-year rule) is arithmetic the model has to do in one
   forward pass with no scratchpad. Rule-consistent augmentation from existing train rows — same
   ticket, shifted `Received` date and restated relative phrase — would multiply the date-reasoning
   signal without inventing new tickets or touching dev/test.
2. **Quoted-email robustness.** Only 38 train rows carry a `>` block, but 11 test rows do. That is
   thin coverage of a rule the spec states explicitly, and it is cheap to augment by splicing
   existing quoted blocks onto existing short tickets, keeping the label unchanged.
3. **Per-field error budget rather than exact match.** `exact_match` hides which field is costing
   the most. The eight-field breakdown above should drive effort allocation.
4. **A second seed on the best config**, to establish whether the Part D gaps are larger than
   run-to-run variance before believing any of them.

## Test predictions

Exactly the graders' contract: verbatim `system` + `user`, Qwen chat template, greedy,
`max_new_tokens=200`, `strip()` as the only post-processing.

In [26]:
t0 = time.time()
test_preds = generate(final_model, test, system=FIXED_SYSTEM, desc="test")
pred_path = os.path.join(OUT_DIR, "predictions_test.jsonl")
with open(pred_path, "w", encoding="utf-8") as f:
    for p in test_preds: f.write(json.dumps(p, ensure_ascii=False) + "\n")
print(f"wrote {len(test_preds)} predictions -> {pred_path}  ({(time.time()-t0)/60:.1f} min)")

ok = sum(1 for p in test_preds if _parse(p["output"]) is not None)
print(f"parse as JSON: {ok}/{len(test_preds)} = {ok/len(test_preds):.1%}")
bad_keys = sum(1 for p in test_preds if (o := _parse(p["output"])) is not None and set(o) != set(KEYS))
print(f"parse but wrong key set: {bad_keys}")
print("\nfirst three:")
for p in test_preds[:3]: print("  ", p["id"], p["output"])

dist = Counter()
for p in test_preds:
    o = _parse(p["output"])
    if o: dist[o.get("category")] += 1
print("\npredicted category mix on test:", dict(dist.most_common()))
print("dev gold category mix          :",
      dict(Counter(json.loads(r['messages'][-1]['content'])['category'] for r in dev).most_common()))

  test 8/400  7s
  test 48/400  42s
  test 88/400  88s
  test 128/400  130s
  test 168/400  176s
  test 208/400  213s
  test 248/400  249s
  test 288/400  285s
  test 328/400  329s
  test 368/400  364s
wrote 400 predictions -> /content/out/predictions_test.jsonl  (6.6 min)
parse as JSON: 400/400 = 100.0%
parse but wrong key set: 0

first three:
   ts-00001 {"category": "fraud", "priority": "P1", "amount_paise": null, "txn_date": "2026-07-00", "txn_id": "NP6090913363", "channel": null, "language": "hi-en", "needs_human": true}
   ts-00002 {"category": "refund", "priority": "P3", "amount_paise": 1302700, "txn_date": "2026-07-29", "txn_id": "NP5221235957", "channel": "card", "language": "hi-en", "needs_human": false}
   ts-00003 {"category": "payment_failed", "priority": "P1", "amount_paise": 61875, "txn_date": "2026-09-11", "txn_id": null, "channel": null, "language": "en", "needs_human": true}

predicted category mix on test: {'payment_failed': 134, 'refund': 91, 'offers': 47, 'account_

In [27]:
# --- reproducibility check: reload the saved adapter from disk and confirm it regenerates
# --- identical output for a random sample. This is the check the graders say they will run.
from peft import PeftModel
free(final_model)
chk = load_base(four_bit=BEST["four_bit"])
chk = PeftModel.from_pretrained(chk, os.path.join(OUT_DIR, "adapter"))
chk.eval()
sample = random.Random(0).sample(test, 12)
again = generate(chk, sample, system=FIXED_SYSTEM, batch_size=6, desc="recheck")
first = {p["id"]: p["output"] for p in test_preds}
mismatch = [p["id"] for p in again if first[p["id"]] != p["output"]]
print(f"\nreloaded-adapter check on {len(sample)} random test rows: "
      f"{len(sample)-len(mismatch)}/{len(sample)} identical")
if mismatch: print("MISMATCH:", mismatch)
else: print("Adapter on disk reproduces predictions_test.jsonl exactly.")
free(chk)

  base loaded, logits finite, 6.27 GB
  recheck 6/12  7s

reloaded-adapter check on 12 random test rows: 12/12 identical
Adapter on disk reproduces predictions_test.jsonl exactly.


In [28]:
# --- copy the submission artifacts together for the repo ---
for fn in ["cleaning_log.csv", "predictions_test.jsonl", "final_metrics.json",
           "experiments_results.json", "baselines.json", "dev_predictions.jsonl"]:
    p = os.path.join(OUT_DIR, fn)
    print(("  ok   " if os.path.isfile(p) else "  MISS ") + p)
print("\nadapter dir:", os.path.join(OUT_DIR, "adapter"))
if os.path.isdir(os.path.join(OUT_DIR, "adapter")):
    for f in sorted(os.listdir(os.path.join(OUT_DIR, "adapter"))):
        fp = os.path.join(OUT_DIR, "adapter", f)
        print(f"   {f:34} {os.path.getsize(fp)/1e6:8.2f} MB")

  ok   /content/out/cleaning_log.csv
  ok   /content/out/predictions_test.jsonl
  ok   /content/out/final_metrics.json
  MISS /content/out/experiments_results.json
  ok   /content/out/baselines.json
  ok   /content/out/dev_predictions.jsonl

adapter dir: /content/out/adapter
   README.md                              0.01 MB
   adapter_config.json                    0.00 MB
   adapter_model.safetensors             73.91 MB
   added_tokens.json                      0.00 MB
   chat_template.jinja                    0.00 MB
   merges.txt                             1.67 MB
   special_tokens_map.json                0.00 MB
   tokenizer.json                        11.42 MB
   tokenizer_config.json                  0.00 MB
   vocab.json                             2.78 MB


In [29]:
# --- bundle everything for download (artifacts are VM-local, so take them now) ---
import shutil, os
zip_base = "/content/nimbuspay_artifacts"
if os.path.isfile(zip_base + ".zip"): os.remove(zip_base + ".zip")
shutil.make_archive(zip_base, "zip", OUT_DIR)
sz = os.path.getsize(zip_base + ".zip") / 1e6
print(f"{zip_base}.zip  {sz:.1f} MB")
for root, _, fs in os.walk(OUT_DIR):
    for f in sorted(fs):
        p = os.path.join(root, f)
        print(f"   {os.path.relpath(p, OUT_DIR):42} {os.path.getsize(p)/1e6:7.2f} MB")
print("\nDownload it from the Files panel on the left (right-click -> Download), or run:")
print("   from google.colab import files; files.download('%s.zip')" % zip_base)

/content/nimbuspay_artifacts.zip  72.4 MB
   baselines.json                                0.00 MB
   cleaning_log.csv                              0.03 MB
   dev_predictions.jsonl                         0.04 MB
   error_attribution.json                        0.00 MB
   final_metrics.json                            0.00 MB
   predictions_test.jsonl                        0.09 MB
   train_clean.jsonl                             1.10 MB
   adapter/README.md                             0.01 MB
   adapter/adapter_config.json                   0.00 MB
   adapter/adapter_model.safetensors            73.91 MB
   adapter/added_tokens.json                     0.00 MB
   adapter/chat_template.jinja                   0.00 MB
   adapter/merges.txt                            1.67 MB
   adapter/special_tokens_map.json               0.00 MB
   adapter/tokenizer.json                       11.42 MB
   adapter/tokenizer_config.json                 0.00 MB
   adapter/vocab.json                         